<a id="introduction"></a>

# PS4: Restricted Boltzmann Machines for Molecular Fingerprints  -  Student

Every FDA-approved drug can be represented as a **binary molecular fingerprint**  -  a bit vector where each bit encodes the presence or absence of a chemical substructure. A 256-bit Morgan fingerprint with radius 2 captures the circular neighborhood up to 2 bonds from each atom. This problem set applies the same RBM tools from L7d to a new domain: drug-like chemistry.

__Why are we looking at this?__ Restricted Boltzmann machines are generative models that learn probability distributions over binary data. By training an RBM on molecular fingerprints, we ask whether the network can capture the latent chemical structure of FDA-approved drugs, reconstruct corrupted fingerprints, and generate novel drug-like bit patterns  -  a simple model of virtual drug discovery.

> __Learning Objectives__
>
> By the end of this problem set, you should be able to:
> * __Represent and compare molecules as binary fingerprints:__ Encode drug structures as 256-bit Morgan fingerprints where each bit captures the presence or absence of a molecular substructure. Implement the Tanimoto similarity coefficient to measure chemical overlap between drug pairs.
> * __Train an RBM on molecular data using contrastive divergence:__ Apply the CD algorithm to learn weights and biases that model the distribution of drug-like fingerprints from a 500-drug training set. Analyze the learned weight patterns to interpret what chemical substructures the hidden units detect.
> * __Use a trained RBM for reconstruction and generative sampling:__ Reconstruct corrupted drug fingerprints via block Gibbs sampling and evaluate recovery quality as a function of corruption fraction. Generate novel virtual drug fingerprints by sampling from random noise and compare their proximity to real FDA-approved drugs using free energy and nearest-neighbor Tanimoto analysis.

Let's get started!
___

<a id="setup"></a>
## Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file and resolving any package conflicts before proceeding.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, and seeds the random number generator for reproducibility.

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl) for RBM operations. Because multiple packages in this environment export a `sample` function, we call RBM sampling as `VLDataScienceMachineLearningPackage.sample(...)` and data sampling as `StatsBase.sample(...)` throughout this notebook.

In [ ]:
# dataset dimensions
n_visible   = 256;    # number of fingerprint bits
n_bits_rows = 16;     # rows for 16×16 visualization
n_bits_cols = 16;     # columns for 16×16 visualization
N_drugs     = 2000;   # total number of drugs in the dataset

# Part 1: exploration
n_display = 6;        # number of fingerprints to display
n_sample  = 100;      # drugs to include in Tanimoto search

# Part 2: small RBM training
n_train    = 500;     # training set size
n_hidden   = 64;      # hidden units for small RBM
η          = 0.005;   # learning rate
β          = 1.0;     # training inverse temperature (keep near 1 for stable CD-k gradients)
β_reconstruct = 2.0;  # inference β for reconstruction and generation
                      #   β shapes the energy landscape during training; β_reconstruct
                      #   sharpens the Gibbs chain at inference so it converges to modes
                      #   decisively rather than drifting.  Use β_reconstruct ≥ β.
n_epochs   = 1000;    # training epochs
batch_size = 32;      # mini-batch size

# Part 3: reconstruction
n_gibbs = 50;         # Gibbs steps for reconstruction

# Part 4: generation
n_generate       = 100;   # virtual fingerprints to generate
n_gibbs_generate = 200;   # Gibbs steps per generated fingerprint
n_show           = 16;    # weight patterns to visualize

### Data
We load 256-bit binary Morgan fingerprints for 2,000 FDA-approved drugs from the `data/` directory. Each fingerprint is stored as a column of a matrix (shape $256 \times N$), where `1.0` indicates a substructure is present and `0.0` indicates absence. Drug class labels are stored in a companion CSV file.

> __Data files__
>
> Both files live in the `data/` directory and are loaded using the `JLD2` and `CSV` packages:
> * `fda_drugs_fingerprints.jld2` stores the `fingerprints` matrix (shape $256 \times 2000$, `Float32`) alongside the `drug_names` vector (`Vector{String}`, length 2000). Each column is a 256-bit Morgan fingerprint for one drug, where `1.0` marks a present substructure and `0.0` marks an absent one. This file is the primary data source for visualization, Tanimoto comparison, and RBM training.
> * `fda_drugs_metadata.csv` records the `name` and `drug_class` for every drug, one row per entry, spanning 20 pharmacological classes. The class labels let us verify that the most Tanimoto-similar pairs belong to the same therapeutic category and that the RBM has learned class-specific structure.

We load both files and confirm the dimensions are consistent.

In [ ]:
fingerprints, drug_names, metadata = let

    # initialize -
    fp_file      = jldopen(joinpath(_PATH_TO_DATA, "fda_drugs_fingerprints.jld2"))
    fingerprints = fp_file["fingerprints"]   # Float32, 256 × N
    drug_names   = fp_file["drug_names"]     # Vector{String}
    close(fp_file)

    metadata = CSV.read(joinpath(_PATH_TO_DATA, "fda_drugs_metadata.csv"), DataFrame)

    println("Loaded: $(size(fingerprints, 2)) drugs, $(size(fingerprints, 1)) bits")
    println("Drug classes: $(length(unique(metadata.drug_class))) classes")
    fingerprints, drug_names, metadata
end;

In [ ]:
let
    println("Sanity checks:")
    println("  fingerprints size : ", size(fingerprints), "  (bits × drugs)")
    println("  drug_names length : ", length(drug_names))
    println("  metadata rows     : ", nrow(metadata))
    println("  mean bits per drug: ", round(mean(sum(fingerprints, dims=1)), digits=1))
    @assert size(fingerprints, 2) == length(drug_names) "column count mismatch"
    @assert nrow(metadata) == length(drug_names) "metadata row count mismatch"
end

### Implementation
There are a few helper functions that we'll define here that will be used throughout the notebook. Let's start with a common _similarity metric_ for binary fingerprints: the Tanimoto coefficient.

> __Tanimoto (Jaccard) similarity__. The Tanimoto coefficient is the standard similarity metric in cheminformatics. For two binary fingerprints $\mathbf{a}, \mathbf{b} \in \{0,1\}^{n}$, it equals the size of the intersection divided by the size of the union of set bits:
>
> $$J(\mathbf{a},\mathbf{b}) = \frac{\langle \mathbf{a},\mathbf{b} \rangle}{\|\mathbf{a}\|_{1} + \|\mathbf{b}\|_{1} - \langle \mathbf{a},\mathbf{b} \rangle}$$
>
> The numerator counts bits set in both fingerprints; the denominator counts bits set in either. Identical fingerprints return $J = 1.0$, fingerprints with no bits in common return $J = 0.0$, and scores above $0.5$ indicate substantial structural overlap.

We implement the Tanimoto coefficient below and verify it on two simple cases.

In [ ]:
function tanimoto(a::AbstractVector, b::AbstractVector)
    ab = dot(a, b)
    return nothing  # TODO: ab / (sum(a) + sum(b) - ab)
end

Let's do a few checks to confirm our `tanimoto(...)` implementation is correct:

In [ ]:
let
    # sanity checks
    v1 = [1.0, 1.0, 0.0, 0.0]
    v2 = [1.0, 1.0, 0.0, 0.0]
    v3 = [0.0, 0.0, 1.0, 1.0]
    
    # what do we expect?
    println("tanimoto(identical):  ", tanimoto(v1, v2))   # expected 1.0
    println("tanimoto(orthogonal): ", tanimoto(v1, v3))   # expected 0.0
end

With `tanimoto(...)` function defined, we now add two more helper functions. In Task 4 we will want to know how closely the RBM's generated fingerprints resemble real FDA drugs, and whether the RBM assigns them high probability.

__Nearest-neighbor search__. The `nn_tanimoto` function returns the maximum Tanimoto similarity between a single query fingerprint and any column of a reference database matrix  -  the similarity to its nearest real-drug neighbor.

In [ ]:
function nn_tanimoto(query::AbstractVector, database::AbstractMatrix)

    # TODO: Return the maximum Tanimoto similarity between query and any column of database.
    # HINT: Loop over j in 1:size(database, 2), compute tanimoto(query, database[:, j])

    # initialize -
    best = -Inf
    for j in 1:size(database, 2)
        s = tanimoto(query, database[:, j])
        s > best && (best = s)
    end
    return nothing  # TODO: return best
end

> __Free energy__. The RBM's free energy $F(\mathbf{v})$ scores how well a fingerprint matches the learned distribution. Lower (more negative) free energy means higher probability under the model. The free energy for a visible state $\mathbf{v}$ in $\pm 1$ encoding is given by:
>
> $$F(\mathbf{v}) = -\langle \mathbf{a},\mathbf{v} \rangle - \sum_{j=1}^{n_{h}} \log\!\left(1 + e^{(\mathbf{W}^{\top}\mathbf{v} + \mathbf{b})_{j}}\right)$$
>
> Lower (more negative) free energy means higher probability under the model. We expect RBM-generated fingerprints to have lower free energy than random bit vectors.

We implement the free energy function and compute it for both sets below.

In [ ]:
function free_energy(v::AbstractVector, model::MyRestrictedBoltzmannMachineModel)
    h_input = model.W' * v .+ model.b
    return nothing  # TODO: -dot(model.a, v) - sum(log.(1.0 .+ exp.(h_input)))
end

___

<a id="task1"></a>

## Task 1: Explore the Drug Fingerprint Dataset

We have pre-computed 256-bit Morgan fingerprints for 2,000 FDA-approved drugs spanning 20 drug classes. In this task we visualize the fingerprints, implement the Tanimoto similarity metric, and identify the most structurally similar drug pair in a random sample.

__Visualize fingerprints__. Each 256-bit fingerprint can be reshaped to a $16 \times 16$ image  -  the same dimensions as the USPS digit images from L7d. White pixels indicate a present substructure; dark pixels indicate absence.

> __What does each image show?__ The spatial arrangement is arbitrary (bits are indexed by hash, not by molecular geometry), but drugs in the same class share a characteristic sparsity pattern because they share core chemical scaffolds encoded in the same bit positions.

We draw `n_display` drugs at random and display their fingerprints side by side.

In [ ]:
let

    # TODO: Visualize n_display drug fingerprints as 16×16 heatmaps.
    # Steps:
    # 1) Sample n_display indices: StatsBase.sample(1:N_drugs, n_display, replace=false)
    # 2) For each index, reshape fingerprints[:, idx] to (n_bits_rows, n_bits_cols)
    # 3) Create a heatmap with the drug name and class as the title
    # 4) Collect plots and display in a (2, 3) layout

    # initialize -
    display_idxs = StatsBase.sample(1:N_drugs, n_display, replace=false)
    fp_plots = []
    for idx in display_idxs
        img  = nothing  # TODO: reshape(fingerprints[:, idx], n_bits_rows, n_bits_cols)
        name = drug_names[idx]
        cls  = metadata.drug_class[idx]
        p = nothing  # TODO: heatmap(img, title="$(name)\n$(cls)", colorbar=false, ...)
        push!(fp_plots, p)
    end
    plot(fp_plots..., layout=(2, 3), size=(900, 400))
end

__Most similar drug pair__. We draw a random sample of `n_sample` drugs and identify the pair with the highest Tanimoto similarity. Drugs in the same class should rank highest because they share core chemical scaffolds.

> __What are we looking for?__ We expect the most similar pair to share the same drug class. A Tanimoto score above $0.5$ indicates substantial structural overlap, while scores above $0.7$ are considered highly similar in cheminformatics practice.

We search all $\binom{n_{\text{sample}}}{2}$ pairs and report the best-scoring pair.

In [ ]:
best_sim, best_i, best_j = let

    # initialize -
    sample_idxs = StatsBase.sample(1:N_drugs, n_sample, replace=false)
    sample_fps  = fingerprints[:, sample_idxs]
    best_sim    = -Inf
    best_pair   = (0, 0)

    for i in 1:n_sample
        for j in (i+1):n_sample
            sim = tanimoto(sample_fps[:, i], sample_fps[:, j])
            if sim > best_sim
                best_sim  = sim
                best_pair = (i, j)
            end
        end
    end

    i, j   = best_pair
    name_i = drug_names[sample_idxs[i]]
    name_j = drug_names[sample_idxs[j]]
    cls_i  = metadata.drug_class[sample_idxs[i]]
    cls_j  = metadata.drug_class[sample_idxs[j]]
    println("Most similar pair (Tanimoto = $(round(best_sim, digits=4))):")
    println("  Drug A: $(name_i)  [$(cls_i)]")
    println("  Drug B: $(name_j)  [$(cls_j)]")
    best_sim, i, j
end;

___

<a id="task2"></a>

## Task 2: Train a Small RBM from Scratch

We train a small RBM ($256$ visible $\rightarrow$ $64$ hidden units) on a 500-drug subset using contrastive divergence with $T=2$ Gibbs steps (CD-1). The API mirrors what we used in L7d.

__Prepare training data__. The fingerprints are stored as $\{0,1\}$ `Float32` values. The RBM operates in the $\{-1,+1\}$ spin encoding. We convert between them with $s = 2v - 1$.

> __Encoding__
>
> We maintain two representations of the fingerprint data throughout the problem set:
> * __Storage encoding__: Each bit $v_i \in \{0,1\}$ is stored as a `Float32`. This encoding is used for Tanimoto similarity, heatmap visualization, and as the reference when measuring reconstruction quality; a value of `1.0` means the molecular substructure is present.
> * __RBM encoding__: The energy function and block Gibbs updates require spin variables $s_i \in \{-1,+1\}$ stored as `Int64`. This symmetric encoding centers the energy landscape at zero and is the format expected by both `learn(...)` and `sample(...)`.
> * __Conversion__: The forward transform $s = 2v - 1$ maps storage bits to spins; the inverse $v = (s+1)/2$ maps back. We apply these conversions at the boundary between data loading and model operations so that each function always receives the encoding it expects.

We randomly select `n_train` drugs from the dataset and prepare both encodings.

In [ ]:
train_idx, fp_train_01, fp_train_pm1 = let

    # TODO: Prepare training data in both {0,1} and {-1,+1} encodings.
    # Steps:
    # 1) Sample n_train indices: train_idx = StatsBase.sample(1:N_drugs, n_train, replace=false)
    # 2) Extract fp_train_01 = fingerprints[:, train_idx]
    # 3) Convert to {-1,+1}: fp_train_pm1 = Int64.(2 .* fp_train_01 .- 1)

    # initialize -
    train_idx    = StatsBase.sample(1:N_drugs, n_train, replace=false)
    fp_train_01  = fingerprints[:, train_idx]            # {0,1} Float32
    fp_train_pm1 = nothing  # TODO: Int64.(2 .* fp_train_01 .- 1)

    println("Training set: $(n_train) drugs, $(n_visible) bits")
    println("Mean bits set: $(round(mean(sum(fp_train_01, dims=1)), digits=1))")
    train_idx, fp_train_01, fp_train_pm1
end;

__Train the RBM__. We initialize the model with small random weights and zero biases, then run `n_epochs` training epochs using contrastive divergence (CD-1). At each epoch we call `learn(...)` for one pass and evaluate the reconstruction error on a 50-drug probe set.

> __Reconstruction error__
>
> After each epoch, we draw 50 training drugs, run `VLDataScienceMachineLearningPackage.sample(...)` for $T=2$ Gibbs steps, convert the result back to $\{0,1\}$ encoding, and compute $1 - J(\mathbf{v}_{\text{orig}}, \mathbf{v}_{\text{rec}})$. A decreasing error confirms the model is learning.

We initialize the model and run the training loop below.

In [ ]:
rbm_small, reconstruction_errors = let

    # TODO: Initialize and train the RBM using contrastive divergence (CD-1).
    # Steps:
    # 1) p_train = Categorical(n_train)
    # 2) build(MyRestrictedBoltzmannMachineModel, (W = 0.01*randn(n_visible, n_hidden), b = zeros(n_hidden), a = zeros(n_visible)))
    # 3) Loop n_epochs; each epoch: rbm = learn(rbm, fp_train_pm1, p_train;
    #       maxnumberofiterations=1, T=2, β=β, batchsize=batch_size, η=η, tol=1e-10, verbose=false)
    # 4) Probe 50 training drugs: run VLDataScienceMachineLearningPackage.sample(rbm, v0; T=n_gibbs, β=β_reconstruct),
    #       convert to {0,1}, compute 1 - tanimoto(...)
    # 5) Return rbm, reconstruction_errors

    # initialize -
    p_train = Categorical(n_train)
    rbm     = nothing  # TODO: build(MyRestrictedBoltzmannMachineModel, (W=..., b=..., a=...))

    reconstruction_errors = Float64[]

    println("Training RBM ($(n_visible) → $(n_hidden)) for $(n_epochs) epochs...")
    for epoch in 1:n_epochs
        rbm = nothing  # TODO: learn(rbm, fp_train_pm1, p_train; maxnumberofiterations=1, T=2, ...)

        # evaluate reconstruction error on probe set using inference settings
        probe_idxs = StatsBase.sample(1:n_train, 50, replace=false)
        err = 0.0
        for k in probe_idxs
            v0     = fp_train_pm1[:, k]
            (V, _) = nothing  # TODO: VLDataScienceMachineLearningPackage.sample(rbm, v0; T=n_gibbs, β=β_reconstruct)
            v_rec  = Float32.((V[:, end] .+ 1) ./ 2)
            err   += 1.0 - tanimoto(fp_train_01[:, k], v_rec)
        end
        push!(reconstruction_errors, err / 50)

        (epoch % 10 == 0) && println("Epoch $(epoch): recon error = $(round(last(reconstruction_errors), digits=4))")
    end

    rbm, reconstruction_errors
end;

The reconstruction error should decrease (or plateau) over training epochs. We plot the convergence curve below.

In [ ]:
let
    plot(1:n_epochs, reconstruction_errors,
         xlabel = "Training Epoch",
         ylabel = "Reconstruction Error  (1 − Tanimoto)",
         title  = "RBM Training Convergence ($(n_visible)→$(n_hidden))",
         label  = "Recon. Error",
         lw = 2, color = :blue, size = (700, 400))
end

__Visualize weight patterns__. Each column $\mathbf{W}[:,k]$ of the weight matrix is a 256-dimensional vector encoding what bit pattern hidden unit $k$ detects. Reshaping each column to $16 \times 16$ produces a feature map analogous to the digit stroke patterns we saw in L7d.

> __What do the weight patterns mean?__ Warm colors (positive weights) indicate bits that excite hidden unit $k$; cool colors (negative weights) indicate bits that suppress it. Structured clusters of warm bits suggest the unit is detecting a chemical scaffold.

We display the first `n_show` hidden-unit weight patterns below.

In [ ]:
let

    # TODO: Visualize the first n_show hidden-unit weight patterns as 16×16 heatmaps.
    # Steps:
    # 1) W = rbm_small.W
    # 2) For k in 1:n_show, reshape W[:, k] to (n_bits_rows, n_bits_cols)
    # 3) Use a diverging colormap (color=:bwr) with symmetric color limits (clim=(-lim, lim))
    # 4) Arrange in a 4×4 grid

    # initialize -
    W = rbm_small.W   # n_visible × n_hidden
    w_plots = []
    for k in 1:n_show
        img = nothing  # TODO: reshape(W[:, k], n_bits_rows, n_bits_cols)
        lim = maximum(abs.(img))
        p   = nothing  # TODO: heatmap(img, colorbar=false, color=:bwr, clim=(-lim,lim), ...)
        push!(w_plots, p)
    end
    plot(w_plots..., layout=(4, 4), size=(700, 700))
end

___

<a id="task3"></a>

## Task 3: Corrupted Drug Recall using a Pre-trained RBM
A larger RBM ($256$ visible $\rightarrow$ $2048$ hidden units) was pre-trained on the full 2,000-drug dataset using CD-10 at β = 1.0 and is stored in `data/pretrained_rbm_drugs.jld2`. We use this model to reconstruct corrupted drug fingerprints and ask whether the reconstruction recovers the correct pharmacological class.

__Load the pre-trained RBM__. We load the saved weight matrix and bias vectors from the `jld2` file and rebuild the `MyRestrictedBoltzmannMachineModel` instance. Reconstruction uses `β_reconstruct` defined in the Setup section.

We rebuild the model below.

In [ ]:
rbm_pretrained = let

    # initialize -
    rbm_file = jldopen(joinpath(_PATH_TO_DATA, "pretrained_rbm_drugs.jld2"))
    rbm = build(MyRestrictedBoltzmannMachineModel, (
        W = rbm_file["W"],
        b = rbm_file["b"],
        a = rbm_file["a"]
    ))
    close(rbm_file)

    n_hidden_pretrained = length(rbm.b)
    println("Pre-trained RBM: $(n_visible) visible → $(n_hidden_pretrained) hidden")
    println("β: $(β)")
    rbm
end;

__Corrupt a held-out drug__. We select a drug that was not in the training set, then flip a fraction of its set bits to `0.0` to simulate a corrupted or incomplete fingerprint.

> __Corruption procedure__
>
> We simulate a corrupted or incomplete fingerprint by zeroing out a fraction of its set bits:
> * We identify all indices where the fingerprint value is `1.0`, i.e., the positions that encode present substructures. Only set bits are candidates for corruption; zeroing an already-zero bit would leave the chemical encoding unchanged.
> * We randomly select `round(Int, corruption_frac * n_set_bits)` of those indices and set them to `0.0`, simulating the loss of detectable substructures as might occur with incomplete spectroscopic data or a partially resolved crystal structure.
> * We convert the corrupted fingerprint from $\{0,1\}$ `Float32` to $\{-1,+1\}$ `Int64` before passing it to the RBM, since `sample(...)` requires spin variables, not binary floats.

We apply 50% corruption to a randomly chosen held-out drug.

In [ ]:
test_drug_idx, fp_orig_01, fp_orig_pm1, fp_corr_01, fp_corr_pm1 = let

    # TODO: Select a held-out drug and corrupt its fingerprint.
    # Steps:
    # 1) held_out_idxs = setdiff(1:N_drugs, train_idx)
    # 2) test_drug_idx = held_out_idxs[rand(1:length(held_out_idxs))]
    # 3) fp_orig_01 = fingerprints[:, test_drug_idx]; fp_orig_pm1 = Int64.(2 .* fp_orig_01 .- 1)
    # 4) Copy fp_orig_01 → fp_corr_01; find set_bits = findall(x -> x == 1.0f0, fp_corr_01)
    # 5) Flip n_flip = round(Int, corruption_frac * length(set_bits)) bits to 0.0f0
    # 6) fp_corr_pm1 = Int64.(2 .* fp_corr_01 .- 1)

    # initialize -
    corruption_frac = 0.50
    held_out_idxs   = setdiff(1:N_drugs, train_idx)
    test_drug_idx   = held_out_idxs[rand(1:length(held_out_idxs))]

    fp_orig_01  = fingerprints[:, test_drug_idx]
    fp_orig_pm1 = Int64.(2 .* fp_orig_01 .- 1)

    println("Test drug: ", drug_names[test_drug_idx])
    println("Class:     ", metadata.drug_class[test_drug_idx])

    fp_corr_01 = copy(fp_orig_01)
    set_bits   = findall(x -> x == 1.0f0, fp_corr_01)
    n_flip     = round(Int, corruption_frac * length(set_bits))
    flip_bits  = StatsBase.sample(set_bits, n_flip, replace=false)
    fp_corr_01[flip_bits] .= 0.0f0
    fp_corr_pm1 = nothing  # TODO: Int64.(2 .* fp_corr_01 .- 1)

    sim_corr = tanimoto(fp_orig_01, fp_corr_01)
    println("Corruption: $(round(Int, corruption_frac*100))%  →  Tanimoto(orig, corrupted) = $(round(sim_corr, digits=4))")

    test_drug_idx, fp_orig_01, fp_orig_pm1, fp_corr_01, fp_corr_pm1
end;

__Reconstruct via Gibbs sampling__. We give the corrupted fingerprint to the pre-trained RBM and run block Gibbs sampling for `n_gibbs` steps at `β_reconstruct`. We use a higher inference β than the training β because the two serve different roles: training β controls how much thermal noise appears in the CD-k gradient estimates (low β = stable gradients), while reconstruction β controls how decisively the Gibbs chain converges to an energy minimum (higher β = sharper activations, chain snaps to modes rather than drifting).

> __Why use β_reconstruct > β for inference?__ After training at β = 1.0, the RBM has learned an energy landscape with wells centered on each drug class. Running Gibbs at β = 1.0 during reconstruction means each unit's activation probability is only moderately peaked — the chain explores broadly and may not converge to the deepest minimum within 50 steps. Raising β_reconstruct to 2.0 sharpens the sigmoid so that moderately strong inputs produce near-certain activations, driving the chain directly into the nearest energy minimum. The energy landscape itself does not change — only the sharpness of the Gibbs updates.

We run the reconstruction and display the original, corrupted, and reconstructed fingerprints side by side.

In [ ]:
let

    # TODO: Reconstruct the corrupted fingerprint and check class recovery.
    # Steps:
    # 1) (V_rec, _) = VLDataScienceMachineLearningPackage.sample(rbm_pretrained, fp_corr_pm1; T=n_gibbs, β=β_reconstruct)
    # 2) fp_rec_pm1 = V_rec[:, end];  fp_rec_01 = Float32.((fp_rec_pm1 .+ 1) ./ 2)
    # 3) Loop over j in 1:N_drugs, compute tanimoto(fp_rec_01, fingerprints[:,j]), track best_sim and best_idx
    # 4) Print original drug name/class, nearest neighbor name/class, and whether class matches
    # 5) Display original, corrupted, and reconstructed as a (1, 3) heatmap grid

    # initialize -
    (V_rec, _) = nothing  # TODO: VLDataScienceMachineLearningPackage.sample(rbm_pretrained, fp_corr_pm1; T=n_gibbs, β=β_reconstruct)
    fp_rec_pm1 = nothing  # TODO: V_rec[:, end]
    fp_rec_01  = nothing  # TODO: Float32.((fp_rec_pm1 .+ 1) ./ 2)

    best_sim, best_idx = -Inf, 0
    for j in 1:N_drugs
        s = nothing  # TODO: tanimoto(fp_rec_01, fingerprints[:, j])
        if s > best_sim
            best_sim, best_idx = s, j
        end
    end

    orig_class = metadata.drug_class[test_drug_idx]
    nn_class   = metadata.drug_class[best_idx]
    println("Original drug:    $(drug_names[test_drug_idx])  ($(orig_class))")
    println("Nearest neighbor: $(drug_names[best_idx])  ($(nn_class))")
    println("Class recovered:  $(orig_class == nn_class)")

    p1 = heatmap(reshape(fp_orig_01, n_bits_rows, n_bits_cols),
                 title="Original",      colorbar=false, color=:greys,
                 aspect_ratio=1, xaxis=false, yaxis=false)
    p2 = heatmap(reshape(fp_corr_01, n_bits_rows, n_bits_cols),
                 title="Corrupted",     colorbar=false, color=:greys,
                 aspect_ratio=1, xaxis=false, yaxis=false)
    p3 = heatmap(reshape(fp_rec_01, n_bits_rows, n_bits_cols),
                 title="Reconstructed", colorbar=false, color=:greys,
                 aspect_ratio=1, xaxis=false, yaxis=false)
    plot(p1, p2, p3, layout=(1, 3), size=(750, 280))
end

__Class recovery rate vs. corruption fraction__. We sweep corruption fractions from 10% to 80% and run 20 trials at each level. For each trial we corrupt a held-out drug, reconstruct it with the pre-trained RBM, find its nearest neighbor in the full database, and check whether that neighbor belongs to the correct pharmacological class.

> __What are we expecting?__ At low corruption (≤ 30%), the surviving class-specific core bits provide a strong signal and the Gibbs chain should reliably converge to the correct drug-class energy well — recovery rate well above 80%. At high corruption (≥ 60%), too few characteristic bits survive to guide the chain to the right mode, and recovery degrades.

We compute the sweep below.

In [ ]:
class_recovery, cf_range = let

    # initialize -
    cf_range      = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
    n_trials      = 20
    held_out_idxs = setdiff(1:N_drugs, train_idx)
    class_recovery = Float64[]

    for cf in cf_range
        n_correct = 0
        for _ in 1:n_trials
            tidx   = held_out_idxs[rand(1:length(held_out_idxs))]
            fp_o   = fingerprints[:, tidx]
            fp_c   = copy(fp_o)
            sbits  = findall(x -> x == 1.0f0, fp_c)
            if !isempty(sbits)
                n_flip = round(Int, cf * length(sbits))
                fbits  = StatsBase.sample(sbits, min(n_flip, length(sbits)), replace=false)
                fp_c[fbits] .= 0.0f0
            end

            fp_c_pm1 = Int64.(2 .* fp_c .- 1)
            (Vr, _)  = nothing  # TODO: VLDataScienceMachineLearningPackage.sample(rbm_pretrained, fp_c_pm1; T=n_gibbs, β=β_reconstruct)
            fp_r     = Float32.((Vr[:, end] .+ 1) ./ 2)

            # find nearest neighbor and check class -
            best_sim, best_idx = -Inf, 0
            for j in 1:N_drugs
                s = tanimoto(fp_r, fingerprints[:, j])
                if s > best_sim
                    best_sim, best_idx = s, j
                end
            end

            if metadata.drug_class[best_idx] == metadata.drug_class[tidx]
                n_correct += 1
            end
        end
        push!(class_recovery, n_correct / n_trials)
    end

    class_recovery, cf_range
end;

We plot class recovery rate (%) against corruption fraction below. A dashed line marks the 80% threshold.

In [ ]:
let
    plot(cf_range, class_recovery .* 100,
         xlabel      = "Corruption Fraction",
         ylabel      = "Class Recovery Rate (%)",
         title       = "Drug Class Recovery vs. Corruption Fraction",
         lw=2, marker=:circle, color=:steelblue, legend=false,
         ylims=(0, 105))
    hline!([70.0], linestyle=:dash, color=:gray, label="70% threshold")
end

___

<a id="task4"></a>

## Task 4: Generative Sampling for Virtual Drug Discovery
The pre-trained RBM defines a probability distribution over 256-bit fingerprints. We can draw samples from this distribution by initializing a random $\pm 1$ state and running Gibbs sampling until the chain mixes. The resulting fingerprints are virtual drugs  -  patterns that the RBM assigns high probability.

__Generate virtual fingerprints__. We initialize `n_generate` random $\pm 1$ visible states and run `n_gibbs_generate` Gibbs steps each at `β`. The final visible state is interpreted as a generated fingerprint.

> __What does convergence mean?__ After enough Gibbs steps, the chain's distribution over visible states converges to the RBM's learned marginal $P(\mathbf{v})$. Patterns that appeared frequently in the training data have lower free energy and are visited more often by the chain.

We generate the virtual fingerprints below.

In [ ]:
generated_01 = let

    # initialize -
    println("Generating $(n_generate) virtual fingerprints ($(n_gibbs_generate) Gibbs steps each)...")
    generated_01 = zeros(Float32, n_visible, n_generate)

    for i in 1:n_generate
        v_noise  = rand([-1, 1], n_visible)
        (Vg, _)  = VLDataScienceMachineLearningPackage.sample(rbm_pretrained, v_noise;
                       T = n_gibbs_generate, β = β_reconstruct)
        generated_01[:, i] = Float32.((Vg[:, end] .+ 1) ./ 2)
    end

    println("Done. Mean bits set: $(round(mean(sum(generated_01, dims=1)), digits=1))")
    generated_01
end;

__Nearest-neighbor Tanimoto__. For each generated fingerprint we find the most similar real FDA drug (nearest neighbor in Tanimoto space). We repeat this for random bit vectors at the same average density to create a baseline.

> __What are we expecting?__ RBM-generated fingerprints should have a higher mean nearest-neighbor Tanimoto than random bit vectors, since the RBM has learned to produce patterns that cluster near real drug fingerprints.

We implement the nearest-neighbor search function and compare RBM samples against random vectors.

In [ ]:
nn_gen, nn_rand, random_01 = let

    # initialize -
    mean_density = mean(sum(fingerprints, dims=1)) / n_visible
    random_01    = Float32.(rand(n_visible, n_generate) .< mean_density)

    nn_gen  = [nn_tanimoto(generated_01[:, i], fingerprints) for i in 1:n_generate]
    nn_rand = [nn_tanimoto(random_01[:, i],    fingerprints) for i in 1:n_generate]

    println("Mean NN-Tanimoto  -  RBM:    $(round(mean(nn_gen),  digits=4))")
    println("Mean NN-Tanimoto  -  Random: $(round(mean(nn_rand), digits=4))")
    nn_gen, nn_rand, random_01
end;

We plot the histogram of nearest-neighbor Tanimoto scores for RBM-generated versus random fingerprints below.

In [ ]:
let
    p = histogram(nn_gen,  bins=20, label="RBM-Generated", alpha=0.6, color=:blue,
                  xlabel="NN Tanimoto to Nearest FDA Drug", ylabel="Count",
                  title="Nearest-Neighbor Tanimoto: Generated vs Random",
                  size=(650, 400))
    histogram!(nn_rand, bins=20, label="Random", alpha=0.6, color=:red)
    p
end

__Free energy comparison__. The nearest-neighbor Tanimoto scores tell us how close the generated fingerprints are to the real drugs in our database, but they do not directly reflect the probability the RBM assigns to those fingerprints. For that we use the `free_energy` function defined in the `### Implementation` section.

We compute the free energy for all `n_generate` generated fingerprints and an equal number of random bit vectors sampled at the same average bit density, then compare their distributions.

In [ ]:
fe_gen, fe_rand = let

    # initialize -
    gen_pm1  = 2.0 .* generated_01 .- 1
    rand_pm1 = 2.0 .* random_01    .- 1

    fe_gen  = [free_energy(gen_pm1[:,  i], rbm_pretrained) for i in 1:n_generate]
    fe_rand = [free_energy(rand_pm1[:, i], rbm_pretrained) for i in 1:n_generate]

    println("Mean Free Energy  -  RBM:    $(round(mean(fe_gen),  digits=2))")
    println("Mean Free Energy  -  Random: $(round(mean(fe_rand), digits=2))")
    fe_gen, fe_rand
end;

We visualize the free energy distributions for RBM-generated and random fingerprints below.

In [ ]:
let
    p = histogram(fe_gen,  bins=20, label="RBM-Generated", alpha=0.6, color=:blue,
                  xlabel="Free Energy F(v)", ylabel="Count",
                  title="Free Energy: Generated vs Random Fingerprints",
                  size=(600, 400))
    histogram!(fe_rand, bins=20, label="Random", alpha=0.6, color=:red)
    p
end

___

<a id="discussion"></a>

## Discussion
We now have a trained small RBM, a pre-trained large RBM, and analyses of reconstruction quality and generative sampling. Use the results above to answer the discussion questions below.

**DQ1: What are the hidden units encoding?** The weight matrix columns of the trained small RBM resemble structured bit patterns when visualized as $16 \times 16$ images.

> __Strategy__: Examine the weight pattern heatmaps from Task 2. Describe in 2–3 sentences what the warm-colored bit clusters suggest about what each hidden unit is detecting, and connect this to the idea of chemical scaffolds or pharmacophore patterns.

In [ ]:
# Answer DQ1 here.

In [ ]:
did_I_answer_DQ1 = false;  # TODO: update to true if answered DQ1 {true | false}

**DQ2: How does corruption fraction affect class recovery?** The sweep in Task 3 shows the fraction of trials in which the reconstructed fingerprint's nearest neighbor belongs to the correct pharmacological class.

> __Strategy__: Identify the approximate corruption fraction at which class recovery drops below 70% on the plot. In 2-3 sentences, explain mechanistically why there is a threshold effect — what happens to the Gibbs chain when too many class-specific core bits are missing?

In [ ]:
# Answer DQ2 here.

In [ ]:
did_I_answer_DQ2 = false;  # TODO: update to true if answered DQ2 {true | false}

**DQ3: What do the free energy distributions reveal about the learned energy landscape?** Task 4 compares the free energy of RBM-generated fingerprints against random binary vectors at the same average bit density.

> __Strategy__: Compare the mean free energies printed in Task 4 and the histogram shapes. In 2–3 sentences, interpret what it means that generated fingerprints have lower (more negative) free energy, and explain why random bit vectors fall at higher energy despite having the same average number of set bits.

In [ ]:
# Answer DQ3 here.

In [ ]:
did_I_answer_DQ3 = false;  # TODO: update to true if answered DQ3 {true | false}

___

<a id="summary"></a>

## Summary
This problem set applied restricted Boltzmann machines to 256-bit molecular fingerprints of FDA-approved drugs, demonstrating that RBMs can learn, reconstruct, and generate drug-like chemical patterns.

> __Key Takeaways__
>
> * **Molecular fingerprints encode chemistry as binary vectors:** A 256-bit Morgan fingerprint maps the 2-bond chemical neighborhood of each atom to a binary bit, enabling computational comparison of drug structures. The Tanimoto coefficient measures the overlap between two fingerprints and serves as the standard molecular similarity metric in cheminformatics.
> * **RBMs learn the latent structure of drug-like chemistry:** Contrastive divergence training updates weights and biases by comparing data-driven and model-reconstructed activations without computing the intractable partition function. After training, each hidden unit's weight vector corresponds to a co-occurring pattern of chemical substructures, acting as a molecular feature detector.
> * **A trained RBM functions as both a reconstructor and a generator:** Block Gibbs sampling recovers corrupted drug fingerprints by projecting noisy inputs back onto the learned data manifold, with Tanimoto recovery above 0.7 for corruption up to roughly 40–50% at inference $\beta=5$. Samples generated from random noise converge to fingerprints with lower free energy than random bit vectors, reflecting the probability landscape the RBM has learned.

The pre-trained RBM provides a starting point for more sophisticated generative models of molecular structure, such as variational autoencoders and graph neural networks.
___

<a id="tests"></a>

## Tests
In the code block below, we check key values computed in your notebook and give you feedback on which items are correct. Run this cell after completing all tasks.

In [ ]:
let
    using Test
    @testset verbose = true "CHEME 5820 PS4 test suite" begin

        @testset "Setup and Data" begin
            @test typeof(fingerprints) == Matrix{Float32}
            @test size(fingerprints) == (n_visible, N_drugs)
            @test length(drug_names) == N_drugs
            @test nrow(metadata) == N_drugs
            @test length(unique(metadata.drug_class)) == 20
        end

        @testset "Task 1: Fingerprint Exploration" begin
            @test isapprox(tanimoto([1.0, 1.0, 0.0, 0.0], [1.0, 1.0, 0.0, 0.0]), 1.0, atol=1e-8)
            @test isapprox(tanimoto([1.0, 0.0, 0.0], [0.0, 1.0, 0.0]), 0.0, atol=1e-8)
            @test 0.0 ≤ best_sim ≤ 1.0
        end

        @testset "Task 2: Train Small RBM" begin
            @test size(rbm_small.W) == (n_visible, n_hidden)
            @test length(rbm_small.b) == n_hidden
            @test length(rbm_small.a) == n_visible
            @test length(reconstruction_errors) == n_epochs
            @test all(isfinite, reconstruction_errors)
            @test last(reconstruction_errors) ≤ first(reconstruction_errors) + 0.15
        end

        @testset "Task 3: Corrupted Drug Recall" begin
            @test size(rbm_pretrained.W, 1) == n_visible
            @test length(fp_orig_01) == n_visible
            @test length(fp_corr_01) == n_visible
            @test isapprox(tanimoto(fp_orig_01, fp_orig_01), 1.0, atol=1e-6)
            @test tanimoto(fp_corr_01, fp_orig_01) < 1.0
            @test length(class_recovery) == 8
            @test all(0.0 .≤ class_recovery .≤ 1.0)
            @test class_recovery[1] > 0.5   # >50% recovery at 10% corruption
        end

        @testset "Task 4: Generative Sampling" begin
            @test size(generated_01) == (n_visible, n_generate)
            @test all(x -> x == 0.0f0 || x == 1.0f0, generated_01)
            @test mean(fe_gen) < mean(fe_rand)
            @test mean(nn_gen) > mean(nn_rand)
        end

        @testset "Discussion Questions" begin
            @test did_I_answer_DQ1 == true
            @test did_I_answer_DQ2 == true
            @test did_I_answer_DQ3 == true
        end

    end
end;